# 06 · LLM Router：TEU 工具、规则预审节点与澄清

用户输入先由 `llm_router` 做结构化意图分析，再选择一条业务分支：

`User input → llm_router → (TEU tool | rule precheck node | clarify) → response_llm → END`

- `calculate_teu`：沿用 05 的 `@tool` TEU 计算能力。
- `rule_precheck_node`：模拟调用规则预审服务，返回可用于课堂演示的 mock 结果。
- `clarify_node`：当箱型、数量或订舱 ID 不足时，为后续 LLM 准备澄清问题。

三条分支都不直接面向用户输出；`response_llm` 根据分支结果统一组织最终回答。


In [ ]:
from __future__ import annotations

import os
from collections import defaultdict
from pathlib import Path

from dotenv import load_dotenv


def find_repo_root() -> Path:
    """向上找到训练仓根目录，避免 Notebook 工作目录变化。"""
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / ".env.example").exists():
            return candidate
    return here


ROOT = find_repo_root()
os.chdir(ROOT)
load_dotenv(ROOT / ".env")

missing_llm = [
    name for name in ("LLM_BASE_URL", "LLM_MODEL")
    if not (os.getenv(name) or "").strip()
]
if missing_llm:
    raise ValueError("真实 LLM Router 必须配置 .env，缺少：" + ", ".join(missing_llm))

print("cwd =", ROOT)
print("mode = live LLM router required")


In [ ]:
def show_graph(graph):
    """在 Notebook 中展示 State / Node / Edge。"""
    from IPython.display import Image, display

    try:
        display(Image(graph.get_graph().draw_mermaid_png()))
    except Exception as exc:
        print("PNG 不可用，打印 Mermaid：", exc)
        print(graph.get_graph().draw_mermaid())


In [ ]:
import json
import operator
from typing import Annotated, Literal, TypedDict

from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from pydantic import BaseModel, Field


RouterIntent = Literal["calculate_teu", "rule_precheck", "clarify"]
RouteKey = Literal["calculate_teu", "rule_precheck", "clarify"]


class RouteDecision(BaseModel):
    """LLM Router 只做意图分类和参数提取。"""

    intent: RouterIntent = Field(description="TEU 计算、规则预审或需要澄清")
    equipment: str | None = Field(default=None, description="TEU 计算的箱型")
    quantity: int | None = Field(default=None, description="TEU 计算的箱数")
    booking_id: str | None = Field(default=None, description="规则预审的订舱 ID")
    clarification_question: str | None = Field(
        default=None,
        description="信息不足时，需要向用户询问的一个问题",
    )


class RouteState(TypedDict, total=False):
    messages: Annotated[list, add_messages]
    router_output: AIMessage
    route: RouteDecision
    selected_route: RouteKey
    business_result: dict
    status: str
    trace: Annotated[list[str], operator.add]


TEU_PER_EQUIPMENT = {"20GP": 1, "40GP": 2, "40HQ": 2}
SERVICE_CALLS = defaultdict(int)


@tool
def calculate_teu(equipment: str, quantity: int) -> int:
    """按课堂约定换算 TEU。支持 20GP、40GP、40HQ。"""
    key = equipment.strip().upper()
    if key not in TEU_PER_EQUIPMENT:
        raise ValueError(f"不支持的箱型: {equipment}")
    if quantity < 0:
        raise ValueError("quantity 不能为负数")
    return TEU_PER_EQUIPMENT[key] * quantity


@tool("rule_precheck")
def select_rule_precheck(booking_id: str) -> str:
    """当用户要求对订舱做规则预审时，选择规则预审节点。"""
    # 该工具只向 Router 提供意图 schema，图中不会直接执行它。
    return booking_id


def mock_rule_precheck_service(booking_id: str) -> dict:
    """模拟外部规则预审服务，课堂环境不发起真实网络请求。"""
    SERVICE_CALLS[booking_id] += 1
    # TODO(training): 返回规则预审服务的 mock 结果，至少包含 booking_id、decision 和 mock 标记。
    raise NotImplementedError("TODO: implement mock rule precheck response")


# TODO(training): 完成 LLM Router 的三分支主流程。
# 1. 创建基础模型，并绑定 calculate_teu 与 rule_precheck 两个意图 schema。
# 2. 实现 llm_router：解析 tool_calls；没有工具调用时生成 clarify 决策。
# 3. 实现 teu_tool_node、rule_precheck_node 与 clarify_node。
# 4. 实现 response_llm，让三条分支统一生成最终回复。
# 5. 实现 route_after_llm 和 ROUTES path_map，并校验必需参数。
# 6. 创建 builder，注册节点并连接：START -> router -> 三分支 -> response_llm -> END。
# 完成后应得到 builder，供下方 builder.compile() 使用。

graph = builder.compile()
show_graph(graph)


In [ ]:
cases = [
    ("请计算 2×40HQ 的 TEU", "calculate_teu", "success"),
    ("请预审订舱 BK-DEMO-006 的适用规则", "rule_precheck", "success"),
    ("请帮我计算 TEU", "clarify", "need_clarification"),
]
results_by_route = {}

for question, expected_route, expected_status in cases:
    result = graph.invoke({"messages": [HumanMessage(content=question)], "trace": []})
    final_answer = result["messages"][-1]
    results_by_route[expected_route] = result

    print("Q:", question)
    print("Router protocol:", result["router_output"].tool_calls or result["router_output"].content)
    print("Router decision:", result["route"].model_dump())
    print("Selected route:", result["selected_route"])
    print("Business result:", result["business_result"])
    print("Trace:", " -> ".join(result["trace"]))
    print("Final answer:", final_answer.content)
    print("---")

    assert result["selected_route"] == expected_route
    assert result["status"] == expected_status
    assert isinstance(final_answer, AIMessage)
    assert str(final_answer.content).strip()
    assert result["trace"][-1] == "response_llm:final"

assert results_by_route["calculate_teu"]["business_result"]["teu"] == 4
assert results_by_route["rule_precheck"]["business_result"]["mock"] is True
assert results_by_route["rule_precheck"]["business_result"]["decision"] == "PASS"
assert results_by_route["clarify"]["business_result"]["type"] == "clarification"
assert SERVICE_CALLS["BK-DEMO-006"] == 1
print("06 LLM Router -> three branches -> response LLM done")


<!-- codex:p0:06 -->
## P0 进阶 · RetryPolicy、异常分类与节点缓存

路由正确不代表外部服务可靠。生产图需要区分：

- `TimeoutError` 等暂态错误：在有限预算内自动重试；
- `ValueError` 等输入错误：立即失败，不应重试；
- 成功且适合复用的只读结果：按输入缓存，并设置 TTL。

主流程中的规则预审节点返回 mock 成功值。为了单独演示 `RetryPolicy` 如何看到真实异常，下面另建一个会抛错的可靠性示例图，不改变主 Router 的三分支路径。

In [ ]:
from langgraph.cache.memory import InMemoryCache
from langgraph.types import CachePolicy, RetryPolicy


class ReliabilityState(TypedDict, total=False):
    booking_id: str
    result: str


RELIABILITY_CALLS = defaultdict(int)


def protected_rule_lookup(state: ReliabilityState) -> dict:
    booking_id = state["booking_id"]
    RELIABILITY_CALLS[booking_id] += 1

    if booking_id == "INVALID":
        raise ValueError("booking_id 格式无效；输入错误不应自动重试")
    if RELIABILITY_CALLS[booking_id] < 3:
        raise TimeoutError(f"transient timeout: {booking_id}")
    return {"result": f"rule-ok:{booking_id}"}


reliability_builder = StateGraph(ReliabilityState)
reliability_builder.add_node(
    "protected_rule_lookup",
    protected_rule_lookup,
    retry_policy=RetryPolicy(
        # 首次失败后等待 0.01 秒再重试；演示中缩短间隔，避免运行太久
        initial_interval=0.01,
        # 每次重试的等待时间乘以该系数；1.0 表示保持固定间隔，不做指数退避
        backoff_factor=1.0,
        # 单次重试最多等待 0.01 秒，防止退避时间无限增长
        max_interval=0.01,
        # 最多尝试 3 次（包含第一次调用），因此最多发生 2 次重试
        max_attempts=3,
        # 关闭随机抖动，让每次等待时间固定，便于课堂观察和测试
        jitter=False,
        # 只有 TimeoutError 才会触发重试；ValueError 等输入错误会立即抛出
        retry_on=TimeoutError,
    ),
    cache_policy=CachePolicy(ttl=60),
)
reliability_builder.add_edge(START, "protected_rule_lookup")
reliability_builder.add_edge("protected_rule_lookup", END)
reliability_app = reliability_builder.compile(cache=InMemoryCache())

first_lookup = reliability_app.invoke({"booking_id": "BK-RETRY-001"})
calls_after_success = RELIABILITY_CALLS["BK-RETRY-001"]
cached_lookup = reliability_app.invoke({"booking_id": "BK-RETRY-001"})

print("first result =", first_lookup["result"])
print("calls after retry success =", calls_after_success)
print("calls after cached invoke =", RELIABILITY_CALLS["BK-RETRY-001"])

assert calls_after_success == 3
assert cached_lookup["result"] == first_lookup["result"]
assert RELIABILITY_CALLS["BK-RETRY-001"] == 3, "第二次相同输入应命中缓存"

try:
    reliability_app.invoke({"booking_id": "INVALID"})
except ValueError as exc:
    print("non-retryable error =", exc)
else:
    raise AssertionError("ValueError 应直接失败")

assert RELIABILITY_CALLS["INVALID"] == 1
print("06 retry classification + cache ok")

<!-- codex:checklist -->
---

## 练习任务 Checklist

完成后逐项勾选：

- [ ] 让 LLM Router 分别输出 `calculate_teu`、`rule_precheck` 与 `clarify` 结构化意图。
- [ ] 确认 TEU 分支调用 `calculate_teu` 工具，规则预审分支返回 mock 服务结果。
- [ ] 准备一个信息不足的输入，验证流程进入澄清节点。
- [ ] 解释 `path_map` 如何把模型决策映射到实际节点。
- [ ] 确认三条分支都汇合到 `response_llm` 生成最终输出。
- [ ] 对 `TimeoutError` 设置有限次数 RetryPolicy，并确认最终调用次数。
- [ ] 证明 `ValueError` 不会被自动重试。
- [ ] 对相同只读输入执行两次，证明第二次命中缓存。

**交付证据：**三条路由 Trace、retry 计数、non-retryable 异常、缓存计数。